# A hole in a Mott insulator: its spin and its charge part ways

In one dimension an electron does not survive as a particle. Take one out of a Hubbard chain and
the hole it leaves splits into a **holon**, which carries its charge, and a **spinon**, which
carries its spin, and the two move at different speeds. This notebook removes an up electron from
the middle of the half-filled chain at $U = 8$ (in units of the hopping),
$c_{c\uparrow}|\mathrm{GS}\rangle$, evolves the
state in time with `quantx_engine.mps`, and follows the charge and the spin the hole took away,
site by site.

The operator changes the charge, so the state moves from the sector $(n_\uparrow, n_\downarrow) =
(L/2, L/2)$ to $(L/2 - 1, L/2)$; as a fermion operator it carries the Jordan–Wigner string. The
notebook checks the dynamics against an exact diagonalisation at 8 sites, then follows the hole at
24 sites, where its charge moves out like a free particle while its spin stays inside the cone of
the spinon velocity from the Bethe ansatz.

It needs the native engine, which the `pip install quantx-engine` wheels carry on Linux, macOS and
Windows, plus SciPy, and runs in about a minute on two CPU threads.

In [ ]:
import itertools
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as sla
import scipy.special as ss

import quantx_engine
from quantx_engine import mps
from quantx_engine.models import HubbardChain

assert quantx_engine.backend_info()["native"], "this notebook needs the native engine"

## 1. Eight sites, against exact diagonalisation

$H = -t_h \sum_{i,\sigma} (c^\dagger_{i\sigma} c_{i+1,\sigma} + \mathrm{h.c.}) + U \sum_i
n_{i\uparrow} n_{i\downarrow}$. The hopping is the unit, $t_h = 1$: energies are in units of it,
times in units of its inverse, and $t$ is the time. `copy().apply("C_up", c)` makes
$c_{c\uparrow}|\mathrm{GS}\rangle$; `evolve` advances it by two-site TDVP, and `expect` gives the
electron density $\langle n_j \rangle$ and the spin $\langle S^z_j \rangle$ of the evolved state,
normalised.

The exact reference builds $H$ on the sectors $(4, 4)$ and $(3, 4)$, 4900 and 3920 states, as sparse
matrices. It orders the modes spin block by spin block (all up, then all down): there hopping
carries no sign, and $c_{c\uparrow}$ the parity of the up electrons left of $c$. The engine orders
them site by site, with the parity of both spins, so the two agree only through the fermion algebra
itself.

In [ ]:
L, U, c, dt, every, n_frames = 8, 8.0, 3, 0.05, 10, 8       # a frame every 0.5, out to 4

def occupations(k):
    # The k-electron occupations of one spin species, as bit masks.
    return [sum(1 << i for i in occ) for occ in itertools.combinations(range(L), k)]

def sector(n_up, n_dn):
    # H on the sector, and the (n_up, n_dn) of every basis state on every site.
    def hop(states):
        index = {s: a for a, s in enumerate(states)}
        rows, cols = [], []
        for a, s in enumerate(states):
            for i in range(L - 1):
                if bin(s & (3 << i)).count("1") == 1:          # one of sites i, i+1 occupied
                    rows.append(index[s ^ (3 << i)])
                    cols.append(a)
        return sp.csr_matrix((-np.ones(len(rows)), (rows, cols)), shape=(len(states),) * 2)
    ups, dns = occupations(n_up), occupations(n_dn)
    H = sp.kron(hop(ups), sp.identity(len(dns))) + sp.kron(sp.identity(len(ups)), hop(dns))
    H = H + sp.diags([U * bin(u & d).count("1") for u in ups for d in dns])
    occ = np.array([[((u >> j) & 1, (d >> j) & 1) for j in range(L)] for u in ups for d in dns])
    return H.tocsr(), occ

def annihilate_up(site, n_up, n_dn):
    # c_{site, up}: the sign counts the up electrons left of the site, the modes ahead of it.
    ups, dns, ups2 = occupations(n_up), occupations(n_dn), occupations(n_up - 1)
    row = {u: a for a, u in enumerate(ups2)}
    M = sp.lil_matrix((len(ups2) * len(dns), len(ups) * len(dns)))
    for a, u in enumerate(ups):
        if (u >> site) & 1:
            sign = (-1) ** bin(u & ((1 << site) - 1)).count("1")
            for b in range(len(dns)):
                M[row[u ^ (1 << site)] * len(dns) + b, a * len(dns) + b] = sign
    return M.tocsr()

H0, _ = sector(4, 4)
H1, occ = sector(3, 4)
E0, g = sla.eigsh(H0, k=1, which="SA")
times = dt * every * np.arange(n_frames + 1)
psi = sla.expm_multiply(-1j * H1, annihilate_up(c, 4, 4) @ g[:, 0], start=0, stop=times[-1],
                        num=n_frames + 1, endpoint=True)
w = np.abs(psi) ** 2
w /= w.sum(axis=1, keepdims=True)
n_ed, sz_ed = w @ occ.sum(axis=2), w @ (0.5 * (occ[:, :, 0] - occ[:, :, 1]))

gs = mps.ground_state(HubbardChain(L=L, U=U), n_up=4, n_dn=4, max_bond=256, cutoff=1e-14,
                      n_sweeps=40, tol=1e-13)
phi = gs.copy().apply("C_up", c)
print("the hole's state is in the sector", phi.sector)
n_mps, sz_mps = [phi.expect("N")], [phi.expect("Sz")]
for _ in range(n_frames):
    phi.evolve(dt=dt, steps=every, max_bond=256, cutoff=1e-14)
    n_mps.append(phi.expect("N"))
    sz_mps.append(phi.expect("Sz"))
err_n, err_s = np.abs(np.array(n_mps) - n_ed).max(), np.abs(np.array(sz_mps) - sz_ed).max()
print(f"E0: MPS {gs.dmrg.energy:.12f}, exact {E0[0]:.12f}")
print(f"largest difference over {n_frames} frames and {L} sites: {err_n:.1e} in <n_j>, "
      f"{err_s:.1e} in <S^z_j>")
assert phi.sector == {"n_up": 3, "n_dn": 4}
assert abs(gs.dmrg.energy - E0[0]) < 1e-10 and max(err_n, err_s) < 1e-6

## 2. Twenty-four sites: the hole splits

The same calls at 24 sites, with bond dimension 64. Removing the electron takes away charge 1 and
spin 1/2: the charge deficit $\delta n_j(t) = \langle n_j \rangle_t - \langle n_j \rangle_\mathrm{GS}$
sums to $-1$, and the spin $\langle S^z_j \rangle_t$ to $-1/2$, at every time.

The charge deficit starts on site $c$. The spin does not: the ground state's antiferromagnetic
correlations decay only as a power law, with alternating sign, so removing an up electron leaves a
staggered spin pattern around it; and at large $U$, where the holon passes, that pattern shifts by
one site. The average over neighbouring sites, $\bar s_j = (\langle S^z_j \rangle +
\langle S^z_{j+1} \rangle)/2$, cancels the staggered part, and what remains is mostly the spin the
hole took away.

In [ ]:
L, U, chi, dt, every, n_frames = 24, 8.0, 64, 0.1, 2, 25    # a frame every 0.2, out to 5
c = L // 2
t0 = time.time()
gs = mps.ground_state(HubbardChain(L=L, U=U), n_up=L // 2, n_dn=L // 2, max_bond=chi)
n0, s0 = gs.expect("N"), gs.expect("Sz")
phi = gs.copy().apply("C_up", c)
dn, s = [phi.expect("N") - n0], [phi.expect("Sz") - s0]
for _ in range(n_frames):
    phi.evolve(dt=dt, steps=every, max_bond=chi)
    dn.append(phi.expect("N") - n0)
    s.append(phi.expect("Sz") - s0)
dn, s = np.array(dn), np.array(s)
times = dt * every * np.arange(n_frames + 1)
x = np.arange(L) - c
s_bar, xm = 0.5 * (s[:, :-1] + s[:, 1:]), x[:-1] + 0.5
print(f"{n_frames * every} steps at chi = {chi} in {time.time() - t0:.0f} s")
print(f"charge removed: {-dn.sum(axis=1).min():.6f} to {-dn.sum(axis=1).max():.6f}; "
      f"spin removed: {-s.sum(axis=1).min():.6f} to {-s.sum(axis=1).max():.6f}")
assert np.allclose(dn.sum(axis=1), -1, atol=1e-6) and np.allclose(s.sum(axis=1), -0.5, atol=1e-6)

v_s = 2 * ss.i1(2 * np.pi / U) / ss.i0(2 * np.pi / U)     # the spinon velocity (section 3)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, data, xs, title in ((axes[0], -dn, x, r"charge removed, $-\delta n_j(t)$"),
                            (axes[1], -s_bar, xm, r"spin removed, $-\bar s_j(t)$")):
    im = ax.pcolormesh(xs, times, data, cmap="magma", shading="nearest", vmin=0)
    ax.plot(2 * times, times, "c--", -2 * times, times, "c--", lw=1)
    ax.plot(v_s * times, times, "w:", -v_s * times, times, "w:", lw=1.5)
    ax.set_xlabel("$j - c$")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
axes[0].set_ylabel("time $t$")
axes[0].plot([], [], "c--", label="$|j - c| = 2t$, the free holon's front")
axes[0].plot([], [], "w:", label=r"$|j - c| = v_s t$, the spinon's")
axes[0].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

## 3. The holon is nearly free; the spinon moves at $v_s$

At $U = \infty$ the hole is a free particle with the hopping $t_h = 1$: the spins only ride along
as it passes (Ogata and Shiba, 1990). A particle released from one site then has the charge
profile $J_{j-c}(2t)^2$, a Bessel function, whose front moves at velocity 2. At $U = 8$ the charge
follows it closely. It starts a little wider: the ground state's virtual hopping, which puts a
doubly occupied site next to an empty one now and then, spreads part of the deficit onto the
neighbours of $c$.

The spinon moves at the spin velocity of the half-filled chain, which the Bethe ansatz gives as
$v_s = 2\, I_1(2\pi/U) / I_0(2\pi/U)$ (see Essler et al., *The One-Dimensional Hubbard Model*,
2005), 0.73 at $U = 8$. As $U \to \infty$ it tends to $\pi J/2$, the spinon velocity of the
Heisenberg chain with $J = 4/U$. By $t = 5$ most of the charge has left the spinon's cone, and most
of the spin has not.

In [ ]:
free = -ss.jv(x[None, :], 2 * times[:, None]) ** 2           # the U = infinity holon
err = np.abs(dn - free).max(axis=1)
print(f"at t = 0 the deficit on site c is {-dn[0, c]:.3f}, and {-dn[0, c - 1] - dn[0, c + 1]:.3f} "
      f"on its neighbours")
print(f"from t = 1 on, the largest |delta n - free holon| is {err[times >= 1].max():.3f}")
assert err[times >= 1].max() < 0.04

cone = v_s * times[:, None] + 1
charge_out = -np.where(np.abs(x)[None, :] > cone, dn, 0).sum(axis=1)
spin_in = -np.where(np.abs(xm)[None, :] <= cone, s_bar, 0).sum(axis=1) / 0.5

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for k in (10, 25):
    axes[0].plot(x, -dn[k], "o", label=f"$t = {times[k]:g}$, MPS")
    axes[0].plot(x, -free[k], "k-", lw=0.8)
axes[0].plot([], [], "k-", lw=0.8, label=r"$J_{j-c}(2t)^2$, $U = \infty$")
axes[0].set_xlabel("$j - c$")
axes[0].set_ylabel(r"$-\delta n_j$")
axes[0].legend(fontsize=8)
axes[1].plot(times, charge_out, label=r"charge outside $|j - c| \leq v_s t + 1$")
axes[1].plot(times, spin_in, label=r"spin inside it")
axes[1].set_xlabel("time $t$")
axes[1].set_ylabel("fraction of the hole's charge or spin")
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"at t = {times[-1]:g}: {charge_out[-1]:.0%} of the hole's charge is outside the spinon's "
      f"cone, |j - c| <= v_s t + 1 = {cone[-1, 0]:.2f}, and {spin_in[-1]:.0%} of its spin inside")
assert charge_out[-1] > 0.6 and spin_in[-1] > 0.8

## What this shows

- **Exact at 8 sites.** The fermion operator, its string and the change of sector are exact: the
  density and the spin of the evolved hole agree with an exact diagonalisation that orders the
  fermions differently, to the precision printed above.
- **Spin–charge separation at 24 sites.** The hole's charge moves out like the free particle it
  is at $U = \infty$, its front at velocity 2, while its spin stays inside the cone of the spinon
  velocity $v_s = 0.73$: by $t = 5$, about three quarters of the charge has left that cone and
  about nine tenths of the spin is still in it.

In runs made separately, the profiles at bond dimension 64 agree with those at bond dimension 128
to $3\times10^{-4}$, and with those at half the time step to $4\times10^{-4}$. Every state above
lived in the engine; the calls moved only their requests and the measured numbers.